In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w8" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
from anthropic.types import Message
BASE = {'id': 'msg_01FAKE', 'type': 'message', 'role': 'assistant', 
'model': 'claude-haiku-4-5', 'stop_sequence': None, 
'usage': {'input_tokens': 0, 'output_tokens': 0}}

ROUNDS = [
    {**BASE, 'stop_reason': 'tool_use', 'content': [
        {'type': 'text', 'text': '규정을 먼저 찾아보겠습니다.'}, 
        {'type': 'tool_use', 'id': 'toolu_01ABC', 'name': 'search_documents', 'input': {
            'query': '부산 출장 숙박비 한도'}}]}, 
    {**BASE, 'stop_reason': 'tool_use', 'content': [
        {'type': 'tool_use', 'id': 'toolu_02DEF', 'name': 'create_draft', 'input': {
            'doc_type': '국내출장 신청서', 
            'fields': {'출장지': '부산', '기간': '10/13~15'}}}]}, 
    {**BASE, 'stop_reason': 'end_turn', 'content': [{'type': 'text', 'text': '신청서 초안을 만들었습니다.'}]}]

CANNED = [Message.model_validate(r) for r in ROUNDS]

for i, m in enumerate(CANNED, 1):
    kinds = [b.type for b in m.content]
    used = [b.name for b in m.content if b.type == 'tool_use']
    print(f"{i}바퀴  {m.stop_reason:<9}  {str(kinds):<22}→ {(used[0] if used else '(도구 없음)')}")


1바퀴  tool_use   ['text', 'tool_use']  → search_documents
2바퀴  tool_use   ['tool_use']          → create_draft
3바퀴  end_turn   ['text']              → (도구 없음)


In [3]:
messages = [{'role': 'user', 'content': '질문 한 줄'}]
for m in CANNED:
    calls = [b for b in m.content if b.type == 'tool_use']
    if not calls:
        break
    messages.append({'role': 'assistant', 'content': [b.type for b in m.content]})
    messages.append({'role': 'user', 'content': ['tool_result'] * len(calls)})
for i, m in enumerate(messages, 1):
    print(i, f"{m['role']:<10}", m['content'])


1 user       질문 한 줄
2 assistant  ['text', 'tool_use']
3 user       ['tool_result']
4 assistant  ['tool_use']
5 user       ['tool_result']


In [4]:
import importlib
from app.services import chat_service as cs

from app.db.session import session_scope
from app.models import Run
from app.services.ids import next_run_id

user = {'name': '김민준', 'dept': '인프라사업부 2팀', 'clearance': '일반'}

with session_scope() as session:
    rid = next_run_id(session)
    session.add(Run(id=rid, user_id=1, question='부산 출장 신청서', mode='mock'))

ours = {'search_documents': {'user': user}, 'create_draft': {'user': user, 'run_id': rid}}

ARGS = {'search_documents': {'query': '부산 출장 숙박비 한도'}, 'create_draft': {'doc_type': '국내출장 신청서', 'fields': {'출장지': '부산', '기간': '10/13~15'}}}

RETURNED = {}
for name, tool_run in cs.TOOL_RUNNERS.items():
    RETURNED[name] = tool_run(**ARGS[name], **ours[name])
    print(f'{name:<17}: {sorted(RETURNED[name])}')

last_out: dict = {}
NAMES = {'search_documents': '검색', 'create_draft': '초안'}
for name in ('search_documents', 'create_draft'):
    last_out = RETURNED[name]
    kept = len(last_out.get('sources', []))
    mark = '   ← ★★★ 사라졌다' if kept == 0 else ''
    print(f'{NAMES[name]} 도구를 돈 뒤 : sources {kept}건{mark}')


AttributeError: module 'app.services.chat_service' has no attribute 'TOOL_RUNNERS'

In [9]:
importlib.reload(cs)
last_out, made = ({}, None)
for name in ('search_documents', 'create_draft'):
    out = RETURNED[name]
    if name == cs.create_draft.NAME:
        made = out
    else:
        last_out = out
print(f"sources   : {len(last_out.get('sources', []))}건   : 살아남았다")
print(f'초안 있음 : {made is not None}')
print(f'초안 칸   : {sorted(made)}')


sources   : 3건   : 살아남았다
초안 있음 : True
초안 칸   : ['body', 'doc_type', 'fields']


In [2]:
from app.tools import registry, search_documents 

registry.load_all() 

for t in registry.all_tools(): 
    print(f"{t.name} - {type(t.side_effect).__name__}  {t.side_effect.value} ")

print("도구 수 : ", len(registry.all_tools()))


search_documents - SideEffect  none 
create_draft - SideEffect  internal 
도구 수 :  2


In [4]:
from app.core.exceptions import ApprovalRequired

from app.tools import registry, send_slack 

t = registry.get("send_slack")
fields = sorted(t.schema["input_schema"]["properties"])
print(f"등급 : {t.side_effect.value} - reversible: {t.reversible} - 모델이 체우는 칸 : {fields}")

try:
    t.run(text="에이전트가 slack 메세지 보내기")
except ApprovalRequired as e:
    print(str(e))
    print(e.status_code, e.code)
    print(e.detail)


등급 : external - reversible: False - 모델이 체우는 칸 : ['text']
'send_slack'은 승인 없이 실행할 수 없습니다.
409 approval_required
approval_service.approve()를 통해서만 실행됩니다.


In [7]:
import importlib
import app.tools.registry as registry
import app.tools.send_slack as send_slack
import app.services.chat_service as cs
from app.core.config import get_settings
from app.core.exceptions import ApprovalRequired

for mod in (registry, send_slack, cs):
    importlib.reload(mod)

get_settings.cache_clear()
s = get_settings()

HAS_KEY = s.is_live and bool(s.anthropic_api_key)
Q = '출장비 기준을 찾아서, 10월 13일부터 15일까지 부산 출장 신청서를 작성하고, 팀 채널에 올려 줘.'
if HAS_KEY:
    out = cs.ask_with_tools(question=Q)
    print('run_id               :', out.run_id)
    print('attempts             :', out.attempts)
    print('draft 있음           :', out.draft is not None)
    print('sent                 :', out.sent)
    print('답에 「보내지/승인」  :', '보내지' in out.answer or '승인' in out.answer)
else:
    print('키가 없는 경로 — 모델이 무엇을 골랐는지는 이 셀로 확인할 수 없습니다')
    registry.load_all()
    try:
        registry.get('send_slack').run(text=Q)
    except ApprovalRequired as e:
        print('셋째 문 :', e.code)
    print('sent 가 채워질 길 :', '없음 — 막힌 결과는 sent 에 담지 않는다')


13:48:42 INFO     app.integrations.llm_claude: ***** answer_with_tools - system : # 사내 규정 안내 도우미 — 도구를 쓰는 길

당신은 회사의 사내 규정을 안내하는 도우미입니다.

## 무엇을 할 수 있나

질문에 답하기 전에 **도구를 불러 근거를 먼저 확보합니다.**
쓸 수 있는 도구는 요청에 함께 실려 옵니다. 이름과 설명을 읽고 필요할 때 부릅니다.

## 언제 도구를 부르나

- 사내 규정·기준·절차·한도를 묻는 질문이면 **먼저 검색합니다.**
- 인사말이나 도구와 상관없는 잡담이면 부르지 않고 그대로 답합니다.
- 한 번 찾아 부족하면 검색어를 바꿔 다시 부를 수 있습니다.
- 밖으로 내보내는 도구는 승인을 거쳐야 실행됩니다. 「승인이 필요하다」는 결과가 오면 **다시 부르지 않고**, 보내지 않았다고 그대로 말합니다.
- 업무 문서(출장 신청서 등)를 작성해 달라는 요청이면 **되묻지 않습니다.** 먼저 검색해 기준을 찾고, 이어서 문서 초안 도구를 부릅니다.
- 질문에 없는 칸(직급 · 출장 목적 등)은 지어내지 않고 「(확인 필요)」로 둡니다. 금액이 직급에 따라 다르면 직급별 기준을 함께 적습니다 — 초안은 사람이 확인하고 고치는 문서입니다.

## 어떻게 답하나

- 한국어 문장으로 답합니다. **JSON 으로 감싸지 않습니다.**
- 도구가 돌려준 근거 안에 있는 내용만 말합니다.
- 근거에 없으면 "규정에서 확인하지 못했습니다" 라고 말합니다. **지어내지 않습니다.**
- 출처를 문장 안에 다시 나열하지 않아도 됩니다. 출처는 시스템이 따로 싣습니다.

## 하지 말 것

- 근거 없이 금액·기간·절차를 말하지 않습니다.
- 문서 번호나 조항 번호를 추측해서 쓰지 않습니다.
- 보안등급·부서를 스스로 판단해 거르려 하지 않습니다. 그 일은 시스템이 이미 했습니다.

## 질문한 사람
김민준 - 인프라사업부 2팀
13:48:42 INFO     app.integratio